# 13 - Linkability Robustness Checks

Este notebook endurece a tarefa de linkability sem mudar a base de dados de segmentos.

Objetivos:
- aumentar a distância entre pares positivos;
- limitar o número de pares positivos por paciente;
- comparar com baselines simples por distância;
- repetir a avaliação com várias seeds.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from config import FINAL_SEGMENT_FEATURES_DIR
from modeling import run_linkability_baselines, run_repeated_linkability_baselines

## Load Data

A análise usa o dataset final escolhido no estudo de segmentação.
Se quiseres uma corrida mais leve, limita o número de chunks.

In [2]:
MAX_CHUNKS = 20

manifest = json.loads((FINAL_SEGMENT_FEATURES_DIR / "manifest.json").read_text(encoding="utf-8"))
chunk_files = [item["chunk_file"] for item in manifest["chunks"]]
if MAX_CHUNKS is not None:
    chunk_files = chunk_files[:MAX_CHUNKS]

features_df = pd.concat(
    [pd.read_csv(FINAL_SEGMENT_FEATURES_DIR / chunk_file, low_memory=False) for chunk_file in chunk_files],
    ignore_index=True,
)

print("Window (s):", manifest["window_sec"])
print("Step (s):", manifest["step_sec"])
print("Chunk files loaded:", len(chunk_files))
print("Features dataframe:", features_df.shape)

Window (s): 2.0
Step (s): 1.0
Chunk files loaded: 20
Features dataframe: (45000, 215)


## Standard Linkability Baseline

Esta é a formulação base já usada no projeto.

In [3]:
standard_linkability = run_linkability_baselines(
    features_df=features_df,
    group_col="patient_id",
    test_size=0.2,
    random_state=42,
    max_pairs=2000,
    representation="absdiff",
    min_segment_gap=0,
)

standard_linkability["summary_df"]

,model,f1_score,f1_macro,balanced_accuracy,roc_auc,pr_auc
0,LogisticRegression,0.982728,0.982750,0.98275,0.998161,0.998167
1,XGBoost,0.983903,0.983999,0.98400,0.999101,0.999114


## Harder Linkability Setting

Aqui endurecemos a tarefa:
- `min_segment_gap = 4` para evitar positivos muito próximos;
- `max_positive_pairs_per_patient = 3` para reduzir redundância;
- inclusão de baselines simples por distância.

In [4]:
harder_linkability = run_linkability_baselines(
    features_df=features_df,
    group_col="patient_id",
    test_size=0.2,
    random_state=42,
    max_pairs=2000,
    representation="absdiff",
    min_segment_gap=4,
    max_positive_pairs_per_patient=3,
    include_distance_baselines=True,
)

harder_linkability["summary_df"]

,model,threshold,f1_score,f1_macro,balanced_accuracy,roc_auc,pr_auc
0,EuclideanDistance,0.05,0.000000,0.333333,0.50000,0.812707,0.867421
1,CosineSimilarity,0.25,0.829690,0.832454,0.83250,0.892479,0.919306
2,LogisticRegression,NaN,0.982482,0.982500,0.98250,0.996604,0.996091
3,XGBoost,NaN,0.983229,0.983250,0.98325,0.998676,0.998730


In [5]:
print("Train pairs:", harder_linkability["train_pair_df"].shape)
print("Test pairs:", harder_linkability["test_pair_df"].shape)
print("Pair feature matrix train:", harder_linkability["X_train"].shape)
print("Pair feature matrix test:", harder_linkability["X_test"].shape)

Train pairs: (4000, 3)
Test pairs: (4000, 3)
Pair feature matrix train: (4000, 208)
Pair feature matrix test: (4000, 208)


## Repeated Group Splits

Esta secção mede a estabilidade da tarefa em várias seeds, sempre com split por paciente.

In [6]:
repeated_linkability = run_repeated_linkability_baselines(
    features_df=features_df,
    random_states=[42, 123, 456],
    group_col="patient_id",
    test_size=0.2,
    max_pairs=2000,
    representation="absdiff",
    min_segment_gap=4,
    max_positive_pairs_per_patient=3,
    include_distance_baselines=True,
)

repeated_linkability["summary_df"]

,model,f1_score_mean,f1_score_std,f1_macro_mean,f1_macro_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,CosineSimilarity,0.818678,0.010045,0.822943,0.010169,0.823083,0.010199,0.884830,0.007503,0.913938,0.005543
1,EuclideanDistance,0.000000,0.000000,0.333333,0.000000,0.500000,0.000000,0.804087,0.007473,0.861535,0.005119
2,LogisticRegression,0.981341,0.002408,0.981333,0.002467,0.981333,0.002466,0.996924,0.000302,0.996567,0.000415
3,XGBoost,0.983195,0.000758,0.983250,0.000750,0.983250,0.000750,0.998672,0.000040,0.998709,0.000018


In [7]:
repeated_linkability["detailed_df"].sort_values(["model", "random_state"]).reset_index(drop=True)

,model,threshold,f1_score,f1_macro,balanced_accuracy,roc_auc,pr_auc,random_state
0,CosineSimilarity,0.25,0.829690,0.832454,0.83250,0.892479,0.919306,42
1,CosineSimilarity,0.25,0.810018,0.812224,0.81225,0.877482,0.908236,123
2,CosineSimilarity,0.30,0.816327,0.824152,0.82450,0.884528,0.914273,456
3,EuclideanDistance,0.05,0.000000,0.333333,0.50000,0.812707,0.867421,42
4,EuclideanDistance,0.05,0.000000,0.333333,0.50000,0.800130,0.859066,123
5,EuclideanDistance,0.05,0.000000,0.333333,0.50000,0.799424,0.858118,456
6,LogisticRegression,NaN,0.982482,0.982500,0.98250,0.996604,0.996091,42
7,LogisticRegression,NaN,0.978575,0.978500,0.97850,0.996963,0.996853,123
8,LogisticRegression,NaN,0.982966,0.983000,0.98300,0.997205,0.996758,456
9,XGBoost,NaN,0.983229,0.983250,0.98325,0.998676,0.998730,42
